# 📄 PagedAttention — vLLM 的 KV Cache 管理深度解析

**本文目标**：深入理解 PagedAttention 的设计思想、数据结构和实现细节。这是 vLLM 最核心的创新。

读完这篇你会理解：
- Block Table 的数据结构和工作原理
- Block 的分配/回收/复制/共享机制
- PagedAttention 如何解决显存碎片问题
- 与操作系统虚拟内存的精确类比

## 1. 从显存碎片到分页管理

### 1.1 传统方案的碎片问题

```
传统静态 KV Cache 分配:

请求 A (max 2048 tokens): [████████████] 预分配 2048 位置
实际在第 300 个 token 结束: [███████░░░░░] → 浪费 1748 位置 (~875 MB)

请求 B (max 4096 tokens):               [███████████████░░░░░░░░░]
                                预分配 4096, 用了 2500 → 浪费 1596 (~798 MB)

请求 C (max 1024 tokens):  [███░░]  预分配 1024, 用了 300 → 浪费 724

总显存: 3 个请求实际用了 ~2.5 GB KV Cache, 但预分配占了 ~5.8 GB
→ 浪费 3.3 GB (57%!)
→ 本来可以服务 7 个请求, 实际只能服务 3 个
```

### 1.2 PagedAttention 的方案

```
PagedAttention Block 管理:

显存物理布局 (Block Pool):
  [B0][B1][B2][B3][B4][B5][B6][B7][B8][B9][B10][B11][B12]...
  每个 Block = 16 tokens (可配置), 大小固定

请求 A (300 tokens = 19 blocks):
  Block Table: [B0] → [B3] → [B7] → ... → [B42]  (19 个 entries)
  → 只分配了 19 blocks = 304 token 位置 (用了 300)
  → 浪费: 4 token 位置 ≈ 2 KB  ← 几乎为 0!

请求 B (2500 tokens = 157 blocks):
  Block Table: [B1] → [B5] → [B9] → ... → [B200] (157 entries)
  → 只分配了 157 blocks = 2512 token 位置 (用了 2500)
  → 浪费: 12 token 位置 ≈ 6 KB

请求 C (300 tokens = 19 blocks):
  Block Table: [B2] → [B6] → [B10] → ... → [B50] (19 entries)

总显存: 3 个请求用了 195 blocks ≈ 3.1 GB (block_size=16)
  vs 传统: ~5.8 GB → 节省 ~47%
  → 可以服务 ~5.5 个这样的请求 (vs 传统 3 个)
```

### 1.3 两个层次的"碎片"

```
PagedAttention 的碎片分析:

  Block 内碎片 (内部碎片):
    每个 block 16 tokens, 请求可能没用完最后一个 block
    平均浪费: ~8 tokens/请求 ≈ 4 KB
    → 几乎可以忽略

  Block 间碎片 (外部碎片):
    Block 统一大小, 分配回收后可能产生外部碎片
    (被回收的 block 散落各处, 不连续)
    → vLLM 使用 free list 管理, 无外部碎片!

  对比:
    传统方案: 内部碎片 = 预分配 - 实际使用 → 巨大
    PagedAttention: 内部碎片 < 1 block → 几乎为 0
```

## 2. 核心数据结构

### 2.1 Block Table

```python
# 概念模型 (基于 vLLM 源码简化)
from typing import List, Optional

class BlockTable:
    """单个 Sequence 的 KV Cache 映射表"""
    block_ids: List[int]  # 物理 block ID 列表
    num_blocks: int       # 已分配的 block 数量
    
    def __getitem__(self, logical_idx: int) -> int:
        """逻辑位置 → 物理 block ID"""
        return self.block_ids[logical_idx]
    
    def append_block(self, physical_block_id: int):
        """追加一个新 block"""
        self.block_ids.append(physical_block_id)
    
    def free_blocks(self, start_idx: int):
        """释放从 start_idx 开始的 blocks"""
        freed = self.block_ids[start_idx:]
        self.block_ids = self.block_ids[:start_idx]
        return freed  # 返回给 free list

# 使用示例:
# 请求生成了 100 个 tokens, block_size=16
# Block Table: [42, 17, 3, 88, 61, 9, 55]  ← 7 blocks
# 物理 block 的 ID 是不连续的 (因为分配/回收)
# 但逻辑上是连续的: token 0-15→block42, token 16-31→block17, ...

# PagedAttention kernel 在做 Attention 时:
# scores = query @ all_keys
# 遍历 block table, 从对应的物理 block 读取 K, V
```

### 2.2 Block Allocator (简化版)

```python
class BlockAllocator:
    """Block 分配器 —— 类比 OS 的物理页分配器"""
    
    def __init__(self, num_blocks: int, block_size: int):
        self.block_size = block_size
        self.free_blocks: List[int] = list(range(num_blocks))
        self.used_blocks: set = set()
    
    def allocate(self, num_blocks: int = 1) -> List[int]:
        """分配 num_blocks 个 block → 返回物理 ID 列表"""
        if len(self.free_blocks) < num_blocks:
            return []  # OOM, 需要抢占或拒绝
        allocated = []
        for _ in range(num_blocks):
            block_id = self.free_blocks.pop()
            self.used_blocks.add(block_id)
            allocated.append(block_id)
        return allocated
    
    def free(self, block_ids: List[int]):
        """释放 blocks → 回到 free list"""
        for bid in block_ids:
            if bid in self.used_blocks:
                self.used_blocks.remove(bid)
                self.free_blocks.append(bid)
    
    def get_num_free_blocks(self) -> int:
        return len(self.free_blocks)

# 与 OS 虚拟内存的精确类比:
#   BlockAllocator  ↔ Physical Frame Allocator
#   BlockTable      ↔ Page Table (per process)
#   block_id        ↔ Physical Frame Number (PFN)
#   logical_idx     ↔ Virtual Page Number (VPN)
#   block_size      ↔ Page Size (通常 4KB)
```

### 2.3 Copy-on-Write (CoW) 用于 Prefix Sharing

```python
# 当多个请求共享同一个前缀 (如 system prompt)
# vLLM 使用 CoW 来避免数据复制

class CopyOnWriteBlockTable:
    """支持 CoW 的 Block Table"""
    
    def share_prefix(self, other: 'CopyOnWriteBlockTable', prefix_len: int):
        """共享 other 的前 prefix_len 个 blocks"""
        n_blocks = (prefix_len + self.block_size - 1) // self.block_size
        for i in range(n_blocks):
            # 共享物理 block: 增加引用计数, 不复制数据
            physical_block = other.block_ids[i]
            self.block_ids.append(physical_block)
            self._inc_refcount(physical_block)
    
    def append_new_token(self, token_kv):
        """追加新 token → 只写自己的 block, 不影响共享的"""
        # 如果当前 block 被多个 Sequence 引用 → CoW 触发
        if self._refcount(self.current_block) > 1:
            # 分配新 block, 复制旧数据
            new_block = self.allocator.allocate(1)[0]
            copy_kv_data(self.current_block, new_block)
            self._dec_refcount(self.current_block)
            self.current_block = new_block
        
        # 写入新 token 的 KV 数据
        self._write_kv(self.current_block, token_kv)
```

## 3. PagedAttention Kernel 的工作原理

### 3.1 Attention 计算中的 Block 遍历

```python
# 伪代码: PagedAttention 的 forward pass
def paged_attention_kernel(query, block_table, k_cache, v_cache, block_size):
    """
    query:       [num_heads, head_dim]     当前 token 的 Q
    block_table: [num_blocks]              逻辑→物理映射
    k_cache:     [num_physical_blocks, block_size, num_kv_heads, head_dim]
    v_cache:     [num_physical_blocks, block_size, num_kv_heads, head_dim]
    """
    output = zeros(num_heads, head_dim)
    
    for logical_idx, physical_block_id in enumerate(block_table):
        # 从 KV Cache 读取当前 block
        k_block = k_cache[physical_block_id]  # [block_size, num_kv_heads, head_dim]
        v_block = v_cache[physical_block_id]  # [block_size, num_kv_heads, head_dim]
        
        # 计算当前 block 的 attention
        scores = query @ k_block.T  # [num_heads, block_size]
        scores = softmax(scores)     # 需要 online softmax (跨 block)
        output += scores @ v_block  # [num_heads, head_dim]
    
    return output
```

### 3.2 关键优化: 减少 HBM 读取

PagedAttention 真正的实现是高度优化的 CUDA kernel:

```
基础循环:
  for each block in block_table:
    load K[block] from HBM → SRAM  (耗时! memory-bound)
    load V[block] from HBM → SRAM  (耗时! memory-bound)
    compute Q @ K.T in SRAM        (快! compute-bound)
    compute softmax in SRAM         (快!)
    compute attn @ V in SRAM       (快!)

优化 1: FlashAttention-style tiling
  不是一次 load 一个 block → 一次 load 多个 blocks
  在 SRAM 内做 block 间的 partial softmax
  → 减少 HBM 访问次数

优化 2: Block-level parallelism
  不同的 block 可以并行处理 (在 GPU 的 SM 间分配)
  → 利用 GPU 的大规模并行性

优化 3: KV Cache 布局优化
  K 和 V 在 HBM 中的存储布局做了 padding/alignment
  → 保证 coalesced memory access (合并访问)
  → 128 bytes 的对齐要求  ← GPU cache line 大小
```

### 3.3 block_size 的选择

```
block_size 的取舍:

block_size = 8:
  优点: 更少的内部碎片 (平均浪费 4 tokens)
  缺点: Block Table 更大 → 更多 metadata 开销
        更多 kernel launch → 更高的调度开销

block_size = 16 (vLLM 默认):
  平衡: 适中的碎片 + 适中的 metadata

block_size = 32:
  优点: 更少的 blocks → 更少的 metadata 和 kernel launch
  缺点: 内部碎片更大 (平均浪费 16 tokens)
        更适合长序列 (Agent/多模态)

实际建议:
  纯文本短对话: block_size=8 or 16
  Agent/长 context: block_size=32 or 64
  vLLM 默认 16 对大多数场景足够好
```

## 4. 代码实验: Block 分配模拟器

In [ ]:
# PagedAttention Block Manager 的 Python 模拟

class BlockManager:
    def __init__(self, num_blocks=1000, block_size=16):
        self.block_size = block_size
        self.free_blocks = list(range(num_blocks))
        self.block_refcount = [0] * num_blocks
    
    def allocate(self, n=1):
        if len(self.free_blocks) < n:
            return None
        return [self.free_blocks.pop() for _ in range(n)]
    
    def free(self, block_ids):
        for bid in block_ids:
            self.block_refcount[bid] -= 1
            if self.block_refcount[bid] == 0:
                self.free_blocks.append(bid)
    
    def inc_ref(self, block_id):
        self.block_refcount[block_id] += 1

class Sequence:
    def __init__(self, seq_id, manager):
        self.seq_id = seq_id
        self.mgr = manager
        self.block_table = []
        self.token_count = 0
    
    def append_tokens(self, n):
        needed = (self.token_count + n + self.mgr.block_size - 1) // self.mgr.block_size
        current = len(self.block_table)
        if needed > current:
            new_blocks = self.mgr.allocate(needed - current)
            if new_blocks is None:
                return False  # OOM
            self.block_table.extend(new_blocks)
        self.token_count += n
        return True
    
    def release(self):
        self.mgr.free(self.block_table)
    
    def wasted_tokens(self):
        """计算内部碎片"""
        capacity = len(self.block_table) * self.mgr.block_size
        return capacity - self.token_count
    
    def share_prefix(self, other_seq, tokens):
        """共享 other_seq 的前缀 (CoW)"""
        n_blocks = (tokens + self.mgr.block_size - 1) // self.mgr.block_size
        for i in range(min(n_blocks, len(other_seq.block_table))):
            self.block_table.append(other_seq.block_table[i])
            self.mgr.inc_ref(other_seq.block_table[i])
        self.token_count = tokens

# === 模拟 ===
print("=" * 60)
print("PagedAttention Block Manager 模拟")
print("=" * 60)

mgr = BlockManager(num_blocks=500, block_size=16)
seqs = []

# 创建 10 个请求, 每个实际长度随机
import random
random.seed(42)

for i in range(10):
    seq = Sequence(i, mgr)
    actual_len = random.randint(50, 500)
    ok = seq.append_tokens(actual_len)
    seqs.append(seq)
    blocks_used = len(seq.block_table)
    wasted = seq.wasted_tokens()
    print(f"  Seq {i}: {actual_len} tokens → {blocks_used} blocks, "
          f"wasted {wasted} tokens ({wasted*0.5:.0f} KB)")

print(f"\n  总 KV Cache: {sum(s.token_count for s in seqs)} tokens ≈ "
      f"{sum(s.token_count for s in seqs)*0.5/1024:.1f} MB")
print(f"  总浪费: {sum(s.wasted_tokens() for s in seqs)} tokens ≈ "
      f"{sum(s.wasted_tokens() for s in seqs)*0.5/1024:.1f} MB")
print(f"  浪费率: {sum(s.wasted_tokens() for s in seqs)/sum(s.token_count for s in seqs)*100:.1f}%")
print(f"  Free blocks: {len(mgr.free_blocks)}")

# 对比传统方案
traditional = sum(500 for s in seqs)  # 传统: 按 max 分配
paged = sum(len(s.block_table) for s in seqs) * 16  # Paged: 按需分配
print(f"\n  传统方案 (每请求按 500 tokens): {traditional} tokens ≈ {traditional*0.5/1024:.1f} MB")
print(f"  PagedAttention:                    {paged} tokens ≈ {paged*0.5/1024:.1f} MB")
print(f"  节省: {(1-paged/traditional)*100:.0f}%")

# 模拟前缀共享
print(f"\n--- 前缀共享 (system prompt) ---")
system = Sequence(99, mgr)
system.append_tokens(2000)  # system prompt

shared_seqs = []
for i in range(5):
    s = Sequence(100+i, mgr)
    s.share_prefix(system, 2000)  # 共享 system prompt
    s.append_tokens(random.randint(100, 300))  # 各自的问题和回答
    shared_seqs.append(s)

total_blocks = len(system.block_table)
for s in shared_seqs:
    total_blocks += len(s.block_table) - (2000 // 16)  # 减去共享的 blocks
print(f"  无共享: 1 + 5 = 6 份 system prompt = {len(system.block_table)*6} blocks")
print(f"  有共享: system x1 + per-seq new blocks = {total_blocks} blocks")
print(f"  节省: {(1 - total_blocks/(len(system.block_table)*6))*100:.0f}%")